In [ ]:
# Define some variables here
_NYT_CONNECTION_FILE = "/Users/yt/Desktop/nyt-connections-rag/data/Connections_Data.csv"
#_NYT_CONNECTION_FILE = '/home/vgupta8/cs429/nyt-connections-rag/data/ConnectionsFinalDataset.json'
_DIFFICULTY = 4
_GAME_DATE = ""
_NUMBER_OF_CONNECTIONS = 5
#_GITHUB_TOKEN = "GITHUB_TOKEN"
#_URL =https://models.inference.ai.azure.com
_GitHUB_TOKEN = "OPENROUTER_API_KEY"
_URL ="https://openrouter.ai/api/v1"

In [ ]:
# Helper functions for wordnet, evaluating connections, and other related tasks

from nltk.corpus import wordnet as wn

def get_word_synsets(words):
    """Return a dict mapping each word to its unique lemma names from WordNet."""
    result = {}
    for word in words:
        synsets = wn.synsets(word.lower().replace(" ", "_"))
        result[word] = list(dict.fromkeys(
            l.name().replace("_", " ") for s in synsets for l in s.lemmas()
        ))
    return result

## Evaluation metrics for predicted groups vs actual answers

def game_completion_rate(all_predicted, all_actual):
    """All 4 groups exactly correct → puzzle is 'completed'."""
    if not all_predicted:
        return 0.0
    completed = 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]
        if all(ps in actual_sets for ps in pred_sets):
            completed += 1
    return completed / len(all_predicted)


def overall_accuracy(all_predicted, all_actual):
    """Proportion of predicted groups that exactly match a ground-truth group."""
    total, correct = 0, 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            total += 1
            if set(pred["words"]) in actual_sets:
                correct += 1
    return correct / total if total else 0.0


def partial_accuracy_rate(all_predicted, all_actual):
    """
    Proportion of predicted groups whose best overlap with any ground-truth group
    is exactly 3 or exactly 2 words (i.e. near-correct but not exact).
    Returns dict: {3: rate_for_3_overlap, 2: rate_for_2_overlap}
    """
    total = 0
    counts = {2: 0, 3: 0}
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            pred_set = set(pred["words"])
            best_overlap = max(len(pred_set & a_set) for a_set in actual_sets)
            total += 1
            if best_overlap in counts:
                counts[best_overlap] += 1
    return {k: v / total if total else 0.0 for k, v in counts.items()}



## Evaluation helpers — run once, then call evaluate_run() in the cells below

from IPython.display import display, HTML

def _badge(text, bg, fg="#fff"):
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:4px;font-size:0.85em;font-weight:bold">{text}</span>')

def render_puzzle(idx, total, contest, raw_reply, predicted, actual, scores):
    correct_groups = scores["correct_groups"]
    completed      = scores["completed"]
    group_acc      = scores["group_acc"]
    actual_sets    = [set(a["words"]) for a in actual]

    status_badge = (_badge("✓ SOLVED", "#2e7d32") if completed
                    else _badge(f"✗ {correct_groups}/4", "#c62828"))

    html = []
    html.append('<div style="border:1px solid #555;border-radius:8px;padding:16px;'
                'margin:12px 0;font-family:monospace">')

    # Header
    html.append('<div style="display:flex;align-items:center;gap:12px;margin-bottom:12px">')
    html.append(f'<b style="font-size:1.1em">[{idx}/{total}] {contest}</b>')
    html.append(status_badge)
    html.append(f'<span style="color:#888;font-size:0.85em">'
                f'acc {group_acc:.0%} | partial-3 {scores["partial_3"]:.0%} | partial-2 {scores["partial_2"]:.0%}'
                f'</span>')
    html.append('</div>')

    # Raw reply (collapsible)
    escaped = raw_reply.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    html.append('<details style="margin-bottom:12px">')
    html.append('<summary style="cursor:pointer;color:#90caf9">▶ Raw model reply</summary>')
    html.append(f'<pre style="background:#1e1e1e;color:#d4d4d4;padding:10px;border-radius:6px;'
                f'overflow-x:auto;white-space:pre-wrap;font-size:0.85em;margin-top:6px">{escaped}</pre>')
    html.append('</details>')

    # Side-by-side table
    html.append('<table style="width:100%;border-collapse:collapse;font-size:0.9em">')
    html.append('<tr>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Predicted</th>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Actual</th>'
                '</tr>')

    for row in range(max(len(predicted), len(actual))):
        html.append('<tr style="vertical-align:top">')

        if row < len(predicted):
            pred = predicted[row]
            ps = set(pred["words"])
            is_correct = ps in actual_sets
            bg = "#1b3a1b" if is_correct else "#3a1a1a"
            icon_color = "#66bb6a" if is_correct else "#ef5350"
            icon = "✓" if is_correct else "✗"
            label = pred.get("answerDescription", "?")
            if is_correct:
                words_display = [f'<span style="color:#66bb6a">{w}</span>' for w in pred["words"]]
                note = ""
            else:
                best_a = max(actual, key=lambda a: len(ps & set(a["words"])))
                best_set = set(best_a["words"])
                words_display = [
                    f'<span style="color:{"#66bb6a" if w in best_set else "#ef5350"}">{w}</span>'
                    for w in pred["words"]
                ]
                overlap = len(ps & best_set)
                note = (f'<div style="color:#888;font-size:0.8em;margin-top:2px">'
                        f'best match: <i>{best_a["answerDescription"]}</i> ({overlap}/4)</div>')
            html.append(
                f'<td style="padding:5px 8px;background:{bg};border-radius:4px">'
                f'<span style="color:{icon_color}">{icon}</span> '
                f'<b style="color:#ccc">{label}</b><br>'
                f'{", ".join(words_display)}{note}</td>'
            )
        else:
            html.append('<td></td>')

        if row < len(actual):
            a = actual[row]
            diff_colors = {"Yellow": "#f9a825", "Green": "#2e7d32",
                           "Blue": "#1565c0", "Purple": "#6a1b9a"}
            cat_color = diff_colors.get(a.get("difficulty", ""), "#555")
            html.append(
                f'<td style="padding:5px 8px;background:#1e1e2e;border-radius:4px">'
                f'<b style="color:{cat_color}">{a["answerDescription"]}</b><br>'
                f'<span style="color:#ccc">{", ".join(a["words"])}</span></td>'
            )
        else:
            html.append('<td></td>')

        html.append('</tr>')

    html.append('</table></div>')
    return "".join(html)


def evaluate_run( raw_replies, all_actual):
    """Evaluate a list of raw model replies against ground-truth answers. Returns per_puzzle_scores."""
    display(HTML(
        f'<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    ))

    predicted_all  = []
    per_puzzle_scores = []

    for i, (raw_reply, actual) in enumerate(zip(raw_replies, all_actual)):
        try:
            s = raw_reply.find('[')
            e = raw_reply.rfind(']') + 1
            predicted = json.loads(raw_reply[s:e])
        except Exception as ex:
            print(f"[{i+1}] Failed to parse JSON: {ex}")
            predicted = []
        predicted_all.append(predicted)

        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]

        correct_groups = sum(1 for ps in pred_sets if ps in actual_sets)
        completed      = int(all(ps in actual_sets for ps in pred_sets) and len(pred_sets) == 4)
        partial_3 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 3
        ) / max(len(pred_sets), 1)
        partial_2 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 2
        ) / max(len(pred_sets), 1)
        group_acc = correct_groups / max(len(pred_sets), 1)

        scores = dict(correct_groups=correct_groups, completed=completed,
                      group_acc=group_acc, partial_3=partial_3, partial_2=partial_2)
        per_puzzle_scores.append({"contest": pool[i]["contest"], **scores})
        display(HTML(render_puzzle(i + 1, len(pool), pool[i]["contest"],
                                   raw_reply, predicted, actual, scores)))

    # Summary table
    n = len(per_puzzle_scores)
    avg_gcr = sum(s["completed"]  for s in per_puzzle_scores) / n
    avg_acc = sum(s["group_acc"]  for s in per_puzzle_scores) / n
    avg_p3  = sum(s["partial_3"]  for s in per_puzzle_scores) / n
    avg_p2  = sum(s["partial_2"]  for s in per_puzzle_scores) / n

    header = (
        '<tr style="color:#aaa;border-bottom:1px solid #555">'
        '<th style="padding:4px 12px;text-align:left">Contest</th>'
        '<th style="padding:4px 12px">Solved</th>'
        '<th style="padding:4px 12px">Group Acc</th>'
        '<th style="padding:4px 12px">Partial-3</th>'
        '<th style="padding:4px 12px">Partial-2</th></tr>'
    )
    rows = "".join(
        f'<tr><td style="padding:4px 12px">{s["contest"]}</td>'
        f'<td style="padding:4px 12px;text-align:center">{"✓" if s["completed"] else "✗"}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["group_acc"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_3"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_2"]:.0%}</td></tr>'
        for s in per_puzzle_scores
    )
    avg_row = (
        f'<tr style="border-top:2px solid #666;font-weight:bold">'
        f'<td style="padding:6px 12px">Average ({n})</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_gcr:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_acc:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p3:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p2:.1%}</td></tr>'
    )
    display(HTML(
        f'<h3 style="font-family:monospace;margin-top:24px">Summary — {n} puzzle{"s" if n!=1 else ""}</h3>'
        f'<table style="border-collapse:collapse;font-family:monospace;font-size:0.9em">'
        f'{header}{rows}{avg_row}</table>'
    ))

    return per_puzzle_scores, dict(gcr=avg_gcr, acc=avg_acc, p3=avg_p3, p2=avg_p2)



In [3]:
# Function for GPT to call WordNet

def analyze_wordnet_similarity(words):
    """Return pairwise WordNet similarity scores for the provided words."""
    similarities = {}
    for i, word1 in enumerate(words):
        for j, word2 in enumerate(words):
            if i < j:
                try:
                    syn1 = wn.synsets(word1.lower())[0] if wn.synsets(word1.lower()) else None
                    syn2 = wn.synsets(word2.lower())[0] if wn.synsets(word2.lower()) else None
                    best_score = 0
                    if syn1 and syn2:
                        candidates = []
                        try:
                            candidates.append(syn1.path_similarity(syn2))
                        except Exception:
                            pass

                        for ic in (globals().get('brown_ic'), globals().get('semcor_ic')):
                            if ic is not None:
                                try:
                                    candidates.append(syn1.res_similarity(syn2, ic))
                                except Exception:
                                    pass
                                try:
                                    candidates.append(syn1.lin_similarity(syn2, ic))
                                except Exception:
                                    pass
                                try:
                                    candidates.append(syn1.jcn_similarity(syn2, ic))
                                except Exception:
                                    pass

                        try:
                            candidates.append(syn1.wup_similarity(syn2))
                        except Exception:
                            pass

                        valid_scores = [score for score in candidates if score is not None]
                        if valid_scores:
                            best_score = max(valid_scores)
                    similarities[f"{word1}-{word2}"] = round(best_score, 3)
                except:
                    similarities[f"{word1}-{word2}"] = 0
    return similarities


def analyze_wordnet_hypernyms(words):
    """Return the top hypernyms for each word."""
    hypernyms = {}
    for word in words:
        try:
            syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
            if syn:
                hypernyms[word] = [h.name().split('.')[0] for h in syn.hypernyms()[:3]]
            else:
                hypernyms[word] = []
        except:
            hypernyms[word] = []
    return hypernyms


def analyze_wordnet_definitions(words):
    """Return the primary WordNet definition for each word."""
    definitions = {}
    for word in words:
        try:
            syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
            if syn:
                definitions[word] = syn.definition()
            else:
                definitions[word] = "No definition found"
        except:
            definitions[word] = "Error getting definition"
    return definitions


def analyze_wordnet_relationships(words):
    """
    Analyze relationships between words using WordNet.

    Args:
        words: List of words to analyze

    Returns:
        Dictionary with all available analysis results.
    """
    return {
        "similarity": analyze_wordnet_similarity(words),
        "hypernyms": analyze_wordnet_hypernyms(words),
        "definitions": analyze_wordnet_definitions(words),
    }

# NYT Connections game data
game_data = {
    "words": [
        "LASER", "PLUCK", "THREAD", "WAX",
        "COIL", "SPOOL", "WIND", "WRAP",
        "HONEYCOMB", "ORGANISM", "SOLAR PANEL", "SPREADSHEET",
        "BALL", "MOVIE", "SCHOOL", "VITAMIN"
    ]
}

In [4]:
import ipywidgets as widgets
from IPython.display import display, Markdown

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
     base_url= _URL,
     api_key=os.getenv(_GitHUB_TOKEN)
   # base_url="https://models.inference.ai.azure.com",
   # api_key=os.getenv("GITHUB_TOKEN")
)



# Function definitions for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_wordnet_relationships",
            "description": "Analyze semantic relationships between words using WordNet",
            "parameters": {
                "type": "object",
                "properties": {
                    "words": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of words to analyze"
                    }
                },
                "required": ["words"]
            }
        }
    }
]

def ask_gpt(messages, prompt, use_tools):
    # Use a fresh local conversation each call, keeping only the system prompt
    local_messages = [messages[0], {"role": "user", "content": prompt}]

    kwargs = {}
    if use_tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=local_messages,
        **kwargs
    )

    if completion.choices and len(completion.choices) > 0:
        message = completion.choices[0].message
        tool_calls = getattr(message, "tool_calls", None)
        print("Tools:", tool_calls)
        if tool_calls:
            # Append the assistant's reply (with tool_calls) to the conversation
            local_messages.append(message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if tool_name == "analyze_wordnet_relationships":
                    tool_result = analyze_wordnet_relationships(**tool_args)
                else:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}

                # tool_call_id is required to match this result to the right call
                local_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })

            followup = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=local_messages
            )
            if followup.choices:
                return followup.choices[0].message.content

        return getattr(message, "content", None)
    return str(completion)




# Randomly select games based on diffculty

### Dataset 2 -- the csv file

In [ ]:
import pandas as pd
import random

df = pd.read_csv(_NYT_CONNECTION_FILE)
pool = []
for game_id, game_df in df.groupby("Game ID"):
    date_str = game_df["Puzzle Date"].iloc[0]
    words = game_df["Word"].tolist()
    answers = []
    for group_name, group_df in game_df.groupby("Group Name"):
        level = group_df["Group Level"].iloc[0]
        answers.append({
            "answerDescription": group_name,
            "words": group_df["Word"].tolist(),
            "difficulty": None
        })
    pool.append({
        "contest": f"NYT Connections - {date_str}",
        "date": date_str,
        "words": words,
        "answers": answers,
        "difficulty": None  # no single difficulty for the whole puzzle
    })
if _NUMBER_OF_CONNECTIONS:
    pool = random.sample(pool, min(NUMBER_OF_CONNECTIONS, len(pool)))
print(f"Selected {len(pool)} puzzle(s) of difficulty {DIFFICULTY} and date '{GAME_DATE}' for evaluation.")
    

Selected 10 puzzle(s) of difficulty 4 and date '' for evaluation.


# RAG Prep - Only Run Once

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import nltk
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

load_dotenv()




True

In [7]:
from nltk.corpus import wordnet as wn

def build_embedding_text(synset):
    definition = synset.definition()
    lemmas = ", ".join([str(lemma.name().replace("_", " ")) for lemma in synset.lemmas()])
    embedding = f"""The definition of {synset.name().split('.', 1)[0]} is {definition}. The words in this synset are: {lemmas}."""

    
    if len(synset.hypernyms()) != 0:
        hypernyms = ", ".join([str(hypernym.name()) for hypernym in synset.hypernyms()])
        embedding += f""" Hypernyms of this synset are: {hypernyms}."""
    if len(synset.examples()) != 0:
        examples = ", ".join([str(f"\"{example}\"") for example in synset.examples()])
        embedding += f""" Examples are: {examples}."""
    return embedding

all_embedding_sentences = []
metadata = []
for synset in wn.all_synsets():
    # if len(synset.lemmas()) >= 2:   # add if database building is slow since a lot of synsets seem to have only one word in them
    all_embedding_sentences.append(build_embedding_text(synset))
    metadata.append({
        "synset": synset.name(),
        "definition": synset.definition(),
        "lemmas": [lemma.name() for lemma in synset.lemmas()],
        "hypernyms":[hypernym.name() for hypernym in synset.hypernyms()],
        "pos": synset.pos()
    })

In [8]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(all_embedding_sentences, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3677 [00:00<?, ?it/s]

In [9]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

normalized_embeddings = normalize(embeddings)
num_dimensions = embeddings.shape[1]
index = faiss.IndexFlatIP(num_dimensions)
index.add(normalized_embeddings) 

faiss.write_index(index, "wordnet.index")

with open("metadata.pkl", "wb") as f:
    pickle.dump(({"documents": all_embedding_sentences, "metadata": metadata}), f)


# RAG Perp - Start here after running above once

In [10]:
# Load FAISS index
index = faiss.read_index("wordnet.index")

# Load documents + metadata
with open("metadata.pkl", "rb") as f:
    data = pickle.load(f)

documents = data["documents"]
metadata = data["metadata"]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

def search(query, k=2):
    
    query_vec = normalize(model.encode([query]))
    distances, indices = index.search(query_vec, k)

    results = []
    for i in indices[0]:
        results.append(documents[i])

    return results

In [15]:
# function for Rag search
def RAG_search(words, k):
    """Return FAISS search results for each pair of words.""" 
    ##probably need a better explanation there, helppppp
    
    pair_results = {}
    for i, word1 in enumerate(words):
        for j, word2 in enumerate(words):
            if i < j:
                results = search(f"{word1} {word2}", k=k)
                pair_results[f"{word1}-{word2}"] = results
    return pair_results

# Baseline Model

In [70]:
#Baseline: no hints provied to GPT
raw_replies_without_hints = []
all_actual_without_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. 

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use your knowledge to find the groups. 
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]  

    # Test GPT with Baseline (no hints)
    print("\nTesting GPT with Baseline (no hints) grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, use_tools=False)
    raw_replies_without_hints.append(raw_reply)
    all_actual_without_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")




Testing GPT with Baseline (no hints) grouping...


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 16384 tokens, but can only afford 9128. To increase, visit https://openrouter.ai/settings/keys and create a key with a higher daily limit', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_396i7pcAxYof3wHeAghb9iXkVAB'}

# Wordnet

In [60]:

#----------------With Wordnet------------------------------
raw_replies_with_hints = []
all_actual_with_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        Use the analyze_wordnet_relationships function to:
        - Find similarities between words
        - Get hypernyms (broader categories) for words
        - Get definitions of words
        The function returns all supported analyses at once.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use WordNet data to support your reasoning.
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]
    
    
    
    #------------------------------------------
    # Test the WordNet integration
    result = analyze_wordnet_relationships(puzzle["words"])
    print("Similarities:", result["similarity"])
    print("Definitions:", result["definitions"])
    print("Hypernyms:", result["hypernyms"])

    # Test GPT with WordNet
    print("\nTesting GPT with WordNet grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, use_tools=True)
    raw_replies_with_hints.append(raw_reply)
    all_actual_with_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


Similarities: {'MICKEY MOUSE-BUG BITE': 0, 'MICKEY MOUSE-HAPPY MEAL': 0, 'MICKEY MOUSE-BARBIE DREAMHOUSE': 0, 'MICKEY MOUSE-LOTTERY TICKET': 0, 'MICKEY MOUSE-GLAD-HAND': 0, 'MICKEY MOUSE-RINKY-DINK': 0, 'MICKEY MOUSE-MERRY-GO-ROUND': 0, 'MICKEY MOUSE-CHERRY BLOSSOM': 0, 'MICKEY MOUSE-TRIVIAL': 0, 'MICKEY MOUSE-SUNNY-SIDE UP': 0, 'MICKEY MOUSE-CALAMINE LOTION': 0, 'MICKEY MOUSE-VINYL RECORD': 0, 'MICKEY MOUSE-FLAMINGO': 0, 'MICKEY MOUSE-TWO-BIT': 0, 'MICKEY MOUSE-YOUR HEAD': 0, 'BUG BITE-HAPPY MEAL': 0, 'BUG BITE-BARBIE DREAMHOUSE': 0, 'BUG BITE-LOTTERY TICKET': 0, 'BUG BITE-GLAD-HAND': 0, 'BUG BITE-RINKY-DINK': 0, 'BUG BITE-MERRY-GO-ROUND': 0, 'BUG BITE-CHERRY BLOSSOM': 0, 'BUG BITE-TRIVIAL': 0, 'BUG BITE-SUNNY-SIDE UP': 0, 'BUG BITE-CALAMINE LOTION': 0, 'BUG BITE-VINYL RECORD': 0, 'BUG BITE-FLAMINGO': 0, 'BUG BITE-TWO-BIT': 0, 'BUG BITE-YOUR HEAD': 0, 'HAPPY MEAL-BARBIE DREAMHOUSE': 0, 'HAPPY MEAL-LOTTERY TICKET': 0, 'HAPPY MEAL-GLAD-HAND': 0, 'HAPPY MEAL-RINKY-DINK': 0, 'HAPPY MEAL-M

# Rag

In [61]:
#----------------With RAG------------------------------
raw_replies_with_RAG = []
all_actual_with_RAG = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    Semantic_info = RAG_search(puzzle["words"], k=2)
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}
        
        You are given pairwise WordNet semantic information. Each key has the format WORD1-WORD2. Each value lists WordNet definitions, synsets, shared senses, synonyms, or hypernyms that may help explain a possible relationship.

        use below semantics infor to support the analysis:
        {json.dumps(Semantic_info, indent=2)}
        Important reasoning rules:
            1. Treat WordNet information as supporting evidence, not as the final answer.
            2. Some WordNet entries may be irrelevant because words can have multiple meanings.
            3. Look for clusters where several words share the same sense, synonym set, hypernym, or category.
            4. A valid group needs all 4 words to fit the same theme, not just one strong pair.
            5. NYT Connections often uses wordplay, idioms, slang, pop culture, fill-in-the-blank phrases, and category names that WordNet may not capture.
            6. Prefer groups where the relationship explains all 4 words naturally.
            7. Every input word must be used exactly once.
            8. No word may appear in more than one group.
            9. Each group must contain exactly 4 words.
       
        
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]    
        """}
        ]
    
    
    
    #------------------------------------------


    # Test GPT with RAG
    print("\nTesting GPT with RAG grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    raw_reply = ask_gpt(messages, group_prompt, use_tools=False)
    raw_replies_with_RAG.append(raw_reply)
    all_actual_with_RAG.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


Testing GPT with RAG grouping...
Tools: None
  [1/10] done: NYT Connections - 2025-11-25

Testing GPT with RAG grouping...
Tools: None
  [2/10] done: NYT Connections - 2025-01-20

Testing GPT with RAG grouping...
Tools: None
  [3/10] done: NYT Connections - 2025-07-18

Testing GPT with RAG grouping...
Tools: None
  [4/10] done: NYT Connections - 2024-12-30

Testing GPT with RAG grouping...
Tools: None
  [5/10] done: NYT Connections - 2023-10-13

Testing GPT with RAG grouping...
Tools: None
  [6/10] done: NYT Connections - 2025-06-21

Testing GPT with RAG grouping...
Tools: None
  [7/10] done: NYT Connections - 2025-09-04

Testing GPT with RAG grouping...
Tools: None
  [8/10] done: NYT Connections - 2025-02-17

Testing GPT with RAG grouping...
Tools: None
  [9/10] done: NYT Connections - 2024-12-07

Testing GPT with RAG grouping...
Tools: None
  [10/10] done: NYT Connections - 2025-09-25


# Baseline Evaluation

In [62]:
## Show Results for Baseline (no hints):
scores_no_hints, summary_no_hints = evaluate_run(
    raw_replies_without_hints,
    all_actual_without_hints,
)

Predicted,Actual
"✗ DISNEY CHARACTERSMICKEY MOUSE, BARBIE DREAMHOUSE, HAPPY MEAL, FLAMINGObest match: THINGS THAT ARE PINK (2/4)","SMALL-TIMEMICKEY MOUSE, RINKY-DINK, TRIVIAL, TWO-BIT"
"✗ IDIOPHONESRINKY-DINK, GLAD-HAND, YOUR HEAD, TWO-BITbest match: SMALL-TIME (2/4)","STARTING WITH OPTIMISTIC WORDSHAPPY MEAL, GLAD-HAND, MERRY-GO-ROUND, SUNNY-SIDE UP"
"✗ MEDICAL TERMSBUG BITE, CALAMINE LOTION, TRIVIAL, SUNNY-SIDE UPbest match: SMALL-TIME (1/4)","THINGS THAT ARE PINKBARBIE DREAMHOUSE, CHERRY BLOSSOM, CALAMINE LOTION, FLAMINGO"
"✗ ENTERTAINMENT TERMSLOTTERY TICKET, VINYL RECORD, MERRY-GO-ROUND, CHERRY BLOSSOMbest match: THINGS YOU CAN SCRATCH (2/4)","THINGS YOU CAN SCRATCHBUG BITE, LOTTERY TICKET, VINYL RECORD, YOUR HEAD"


Predicted,Actual
"✗ BODY PARTSSHOULDER, ELBOW, FOREARM, ANGLEbest match: CORNERS (2/4)","ASSOCIATED WITH POPEYESPINACH, ANCHOR, PIPE, FOREARM"
"✗ COOKING INGREDIENTSCOOKIE, SPINACH, CHEAT, HANDLEbest match: ___ SHEET (2/4)","CORNERSCROOK, ELBOW, BEND, ANGLE"
"✗ TOOLSPIPE, ANCHOR, CROOK, BENDbest match: ASSOCIATED WITH POPEYE (2/4)","TAKE ON, AS A RESPONSIBILITYBEAR, SHOULDER, HANDLE, ASSUME"
"✗ MISCELLANEOUS TERMSBEAR, FITTED, ASSUME, BALANCEbest match: TAKE ON, AS A RESPONSIBILITY (2/4)","___ SHEETCOOKIE, CHEAT, FITTED, BALANCE"


Predicted,Actual
"✗ VERBSSEE, SPOT, RUN, CATCHbest match: PICK UP ON (3/4)","ELECTRIC ___SLIDE, GUITAR, BLANKET, EEL"
"✗ ANIMALSCOBRA, EEL, COW, STRINGbest match: YOGA BACKBENDS (2/4)","PICK UP ONSEE, SPOT, CATCH, NOTE"
"✗ MUSICAL TERMSGUITAR, NOTE, SLIDE, STREAKbest match: ELECTRIC ___ (2/4)","SEQUENCERUN, STRING, STREAK, SERIES"
"✗ STRUCTURES/OBJECTSBLANKET, BRIDGE, WHEEL, SERIESbest match: YOGA BACKBENDS (2/4)","YOGA BACKBENDSCOBRA, BRIDGE, COW, WHEEL"


Predicted,Actual
"✗ COLORSPINK, WATER, MELT, COUGHbest match: LUNCH ORDERS (1/4)","LUNCH ORDERSCLUB, HERO, WRAP, MELT"
"✗ GAMESPONY, CLUB, ANTE, JEOPARDYbest match: PAY, WITH “UP” (2/4)","NAMES FEATURING ""!""PINK, AIRPLANE, JEOPARDY, YAHOO"
"✗ FOODSGRINDER, BEANS, WRAP, HERObest match: LUNCH ORDERS (2/4)","PAY, WITH “UP”PONY, ANTE, SETTLE, COUGH"
"✗ TRANSPORTATIONAIRPLANE, YAHOO, FILTER, SETTLEbest match: NAMES FEATURING ""!"" (2/4)","USED TO MAKE COFFEEGRINDER, WATER, BEANS, FILTER"


Predicted,Actual
"✓ POETIC ELEMENTSLINE, VERSE, RHYME, METER","BEANSLIMA, PINTO, KIDNEY, FAVA"
"✗ CITIESLIMA, LAGOS, LUXOR, LINCOLNbest match: CITIES BEGINNING WITH “L” (3/4)","CITIES BEGINNING WITH “L”LIMERICK, LINCOLN, LAGOS, LUXOR"
"✗ FOODSPINTO, FAVA, KIDNEY, DUDEbest match: BEANS (3/4)","POETRY TERMSLINE, VERSE, RHYME, METER"
"✗ CREATIVESCREATOR, RAPPER, STALLION, LIMERICKbest match: “THE(E) ___” RAPPERS (3/4)","“THE(E) ___” RAPPERSCREATOR, RAPPER, STALLION, DUDE"


...

Predicted,Actual
"✗ CANDY & SWEETSHONEYCOMB, CANDY CANE, PAMPLEMOUSSE, CORNICHONbest match: FRENCH FOOD WORDS (2/4)","CLEAN UP, AS A PHOTOGRAPHAIRBRUSH, FIX, TOUCH UP, PHOTOSHOP"
"✓ PHOTO EDITINGAIRBRUSH, TOUCH UP, PHOTOSHOP, FIX","FRENCH FOOD WORDSPAIN, PAMPLEMOUSSE, VINAIGRETTE, CORNICHON"
"✗ LIGHT DECORATIONSSTRING LIGHTS, TINSEL, ANGEL, SOLAR PANELbest match: WHAT YOU MIGHT SEE ON A CHRISTMAS TREE (3/4)","THINGS WITH CELLSHONEYCOMB, ORGANISM, SOLAR PANEL, SPREADSHEET"
"✗ CONDIMENTS & DRESSINGSVINAIGRETTE, PAIN, ORGANISM, SPREADSHEETbest match: FRENCH FOOD WORDS (2/4)","WHAT YOU MIGHT SEE ON A CHRISTMAS TREECANDY CANE, ANGEL, TINSEL, STRING LIGHTS"


Predicted,Actual
"✗ TYPES OF STRIKESBLOW, STRIKE, BREAK, CURVEbest match: BOWLING RESULTS (1/4)","BOWLING RESULTSDOUBLE, STRIKE, SPARE, TURKEY"
"✗ LEAVE TAKINGVACATION, SPARE, REST, LEAVEbest match: TIME OFF (3/4)","FOLLOW A MEANDERING COURSESNAKE, WIND, WEAVE, CURVE"
"✗ FORMS OF SNAKESSWORD, TURKEY, CAT, DOUBLEbest match: BOWLING RESULTS (2/4)","TIME OFFBREAK, LEAVE, REST, VACATION"
"✗ MOTIONS OR MOVEMENTSWIND, WEAVE, SNAKE, GOLDbest match: FOLLOW A MEANDERING COURSE (3/4)","___FISHBLOW, CAT, SWORD, GOLD"


Predicted,Actual
"✗ PARTS OF AN INSECTWINGS, LEGS, HORN, FLYbest match: LONG___ (2/4)","LONG___HORN, LEGS, FELLOW, BOW"
"✗ FRIENDSCHUM, FELLOW, ASSOCIATE, AFFILIATEbest match: THINK OF TOGETHER (2/4)","THINK OF TOGETHERASSOCIATE, RELATE, EQUATE, AFFILIATE"
"✓ VERBS OF RELATIONSHIPRELATE, EQUATE, ASSOCIATE, AFFILIATE","W.N.B.A. TEAMSSUN, WINGS, STORM, LIBERTY"
"✗ NATURE ELEMENTSSUN, STORM, LIBERTY, BAITbest match: W.N.B.A. TEAMS (3/4)","WAYS TO ATTRACT FISHFLY, CHUM, BAIT, LURE"


Predicted,Actual
"✗ COLORSBLUE, CONE, CUP, SHAKEbest match: ICE CREAM PARLOR ORDERS (3/4)","ICE CREAM PARLOR ORDERSSPLIT, CUP, SHAKE, CONE"
"✗ POLITICAL TERMSPROGRESSIVE, LIBERAL, LEFT, DEPARTEDbest match: LEFT-LEANING, POLITICALLY (3/4)","LEFT-LEANING, POLITICALLYLEFT, PROGRESSIVE, BLUE, LIBERAL"
"✗ FICTIONAL CHARACTERSMARTIAN, FICTIONAL BOXER, GREEK/ROMAN GOD, RAINMAKERbest match: MATT DAMON MOVIES, WITH ""THE"" (2/4)","MATT DAMON MOVIES, WITH ""THE""MARTIAN, DEPARTED, RAINMAKER, GOOD SHEPHERD"
"✗ THEATER TERMSSPACECRAFT, SPLIT, THEATER, GOOD SHEPHERDbest match: NAMED ""APOLLO"" (2/4)","NAMED ""APOLLO""GREEK/ROMAN GOD, SPACECRAFT, THEATER, FICTIONAL BOXER"


Contest,Solved,Group Acc,Partial-3,Partial-2
NYT Connections - 2025-11-25,✗,0%,0%,75%
NYT Connections - 2025-01-20,✗,0%,0%,100%
NYT Connections - 2025-07-18,✗,0%,25%,100%
NYT Connections - 2024-12-30,✗,0%,0%,75%
NYT Connections - 2023-10-13,✗,25%,100%,100%
NYT Connections - 2025-06-21,✗,0%,0%,100%
NYT Connections - 2025-09-04,✗,25%,50%,100%
NYT Connections - 2025-02-17,✗,0%,50%,75%
NYT Connections - 2024-12-07,✗,25%,50%,100%
NYT Connections - 2025-09-25,✗,0%,50%,100%


In [23]:
# Wordnet Evaluation

In [63]:
## Show Results for WordNet:
scores_with_hints, summary_with_hints = evaluate_run(
    raw_replies_with_hints,
    all_actual_with_hints
)

Predicted,Actual
"✗ DISNEY CHARACTERS AND MERCHANDISEMICKEY MOUSE, HAPPY MEAL, BARBIE DREAMHOUSE, MERRY-GO-ROUNDbest match: STARTING WITH OPTIMISTIC WORDS (2/4)","SMALL-TIMEMICKEY MOUSE, RINKY-DINK, TRIVIAL, TWO-BIT"
"✗ UNIMPORTANT OR TRIVIAL TERMSLOTTERY TICKET, GLAD-HAND, RINKY-DINK, TRIVIALbest match: SMALL-TIME (2/4)","STARTING WITH OPTIMISTIC WORDSHAPPY MEAL, GLAD-HAND, MERRY-GO-ROUND, SUNNY-SIDE UP"
"✗ FIRST AID AND FOOD RELATED TERMSBUG BITE, CALAMINE LOTION, SUNNY-SIDE UP, YOUR HEADbest match: THINGS YOU CAN SCRATCH (2/4)","THINGS THAT ARE PINKBARBIE DREAMHOUSE, CHERRY BLOSSOM, CALAMINE LOTION, FLAMINGO"
"✗ NATURE AND CULTURE RELATED TERMSCHERRY BLOSSOM, VINYL RECORD, FLAMINGO, TWO-BITbest match: THINGS THAT ARE PINK (2/4)","THINGS YOU CAN SCRATCHBUG BITE, LOTTERY TICKET, VINYL RECORD, YOUR HEAD"


Predicted,Actual
"✗ BAKED GOODSCOOKIE, BEAR, SPINACH, SHOULDERbest match: TAKE ON, AS A RESPONSIBILITY (2/4)","ASSOCIATED WITH POPEYESPINACH, ANCHOR, PIPE, FOREARM"
"✗ BODY PARTSCROOK, ANCHOR, ELBOW, CHEATbest match: CORNERS (2/4)","CORNERSCROOK, ELBOW, BEND, ANGLE"
"✗ MECHANICAL COMPONENTSPIPE, HANDLE, BEND, FOREARMbest match: ASSOCIATED WITH POPEYE (2/4)","TAKE ON, AS A RESPONSIBILITYBEAR, SHOULDER, HANDLE, ASSUME"
"✗ MATHEMATICAL TERMSFITTED, ANGLE, ASSUME, BALANCEbest match: ___ SHEET (2/4)","___ SHEETCOOKIE, CHEAT, FITTED, BALANCE"


Predicted,Actual
"✗ ACTIONSSEE, SPOT, RUN, CATCHbest match: PICK UP ON (3/4)","ELECTRIC ___SLIDE, GUITAR, BLANKET, EEL"
"✗ MUSICAL ITEMSCOBRA, SLIDE, GUITAR, STRINGbest match: ELECTRIC ___ (2/4)","PICK UP ONSEE, SPOT, CATCH, NOTE"
"✗ ANIMALS/STRUCTURESBLANKET, BRIDGE, EEL, COWbest match: ELECTRIC ___ (2/4)","SEQUENCERUN, STRING, STREAK, SERIES"
"✗ SEQUENCESSTREAK, NOTE, WHEEL, SERIESbest match: SEQUENCE (2/4)","YOGA BACKBENDSCOBRA, BRIDGE, COW, WHEEL"


Predicted,Actual
"✗ COLORPINK, PONY, CLUB, AIRPLANEbest match: NAMES FEATURING ""!"" (2/4)","LUNCH ORDERSCLUB, HERO, WRAP, MELT"
"✗ FOODGRINDER, WATER, ANTE, HERObest match: USED TO MAKE COFFEE (2/4)","NAMES FEATURING ""!""PINK, AIRPLANE, JEOPARDY, YAHOO"
"✗ RISKWRAP, SETTLE, JEOPARDY, BEANSbest match: LUNCH ORDERS (1/4)","PAY, WITH “UP”PONY, ANTE, SETTLE, COUGH"
"✗ ACTIONYAHOO, FILTER, MELT, COUGHbest match: LUNCH ORDERS (1/4)","USED TO MAKE COFFEEGRINDER, WATER, BEANS, FILTER"


Predicted,Actual
"✗ POETIC FORMSLINE, LIMERICK, VERSE, RHYMEbest match: POETRY TERMS (3/4)","BEANSLIMA, PINTO, KIDNEY, FAVA"
"✗ ANIMALSSTALLION, PINTO, KIDNEY, LIMAbest match: BEANS (3/4)","CITIES BEGINNING WITH “L”LIMERICK, LINCOLN, LAGOS, LUXOR"
"✗ HUMAN BEINGSCREATOR, LINCOLN, RAPPER, DUDEbest match: “THE(E) ___” RAPPERS (3/4)","POETRY TERMSLINE, VERSE, RHYME, METER"
"✗ LOCATIONSLUXOR, LAGOS, FAVA, METERbest match: CITIES BEGINNING WITH “L” (2/4)","“THE(E) ___” RAPPERSCREATOR, RAPPER, STALLION, DUDE"


...

Predicted,Actual
"✗ FOOD ITEMSHONEYCOMB, CANDY CANE, PAMPLEMOUSSE, CORNICHONbest match: FRENCH FOOD WORDS (2/4)","CLEAN UP, AS A PHOTOGRAPHAIRBRUSH, FIX, TOUCH UP, PHOTOSHOP"
"✗ EMOTIONAL STATESPAIN, ANGEL, FIX, TOUCH UPbest match: CLEAN UP, AS A PHOTOGRAPH (2/4)","FRENCH FOOD WORDSPAIN, PAMPLEMOUSSE, VINAIGRETTE, CORNICHON"
"✗ ARTISTIC TOOLSAIRBRUSH, PHOTOSHOP, VINAIGRETTE, TINSELbest match: CLEAN UP, AS A PHOTOGRAPH (2/4)","THINGS WITH CELLSHONEYCOMB, ORGANISM, SOLAR PANEL, SPREADSHEET"
"✗ TECHNOLOGICAL SYSTEMSORGANISM, SOLAR PANEL, SPREADSHEET, STRING LIGHTSbest match: THINGS WITH CELLS (3/4)","WHAT YOU MIGHT SEE ON A CHRISTMAS TREECANDY CANE, ANGEL, TINSEL, STRING LIGHTS"


Predicted,Actual
"✗ SPORTS TERMSDOUBLE, BLOW, STRIKE, SPAREbest match: BOWLING RESULTS (3/4)","BOWLING RESULTSDOUBLE, STRIKE, SPARE, TURKEY"
"✗ ANIMALSSNAKE, CAT, TURKEY, GOLDbest match: ___FISH (2/4)","FOLLOW A MEANDERING COURSESNAKE, WIND, WEAVE, CURVE"
"✗ VACATION ACTIVITIESBREAK, LEAVE, WIND, VACATIONbest match: TIME OFF (3/4)","TIME OFFBREAK, LEAVE, REST, VACATION"
"✗ TEXTILE ACTIONSREST, SWORD, WEAVE, CURVEbest match: FOLLOW A MEANDERING COURSE (2/4)","___FISHBLOW, CAT, SWORD, GOLD"


Predicted,Actual
"✗ NATURE ELEMENTSSUN, FLY, HORN, WINGSbest match: W.N.B.A. TEAMS (2/4)","LONG___HORN, LEGS, FELLOW, BOW"
"✗ SOCIAL CONNECTIONSASSOCIATE, LEGS, FELLOW, BOWbest match: LONG___ (3/4)","THINK OF TOGETHERASSOCIATE, RELATE, EQUATE, AFFILIATE"
"✗ PERSONAL RELATIONSHIPSSTORM, CHUM, RELATE, LIBERTYbest match: W.N.B.A. TEAMS (2/4)","W.N.B.A. TEAMSSUN, WINGS, STORM, LIBERTY"
"✗ ENTICEMENTSBAIT, EQUATE, LURE, AFFILIATEbest match: THINK OF TOGETHER (2/4)","WAYS TO ATTRACT FISHFLY, CHUM, BAIT, LURE"


Predicted,Actual
"✗ SPACE RELATEDGREEK/ROMAN GOD, MARTIAN, LEFT, SPACECRAFTbest match: NAMED ""APOLLO"" (2/4)","ICE CREAM PARLOR ORDERSSPLIT, CUP, SHAKE, CONE"
"✗ THEATER TERMINOLOGYSPLIT, CUP, DEPARTED, THEATERbest match: ICE CREAM PARLOR ORDERS (2/4)","LEFT-LEANING, POLITICALLYLEFT, PROGRESSIVE, BLUE, LIBERAL"
"✗ MARKETING SPEECHPROGRESSIVE, SHAKE, FICTIONAL BOXER, RAINMAKERbest match: ICE CREAM PARLOR ORDERS (1/4)","MATT DAMON MOVIES, WITH ""THE""MARTIAN, DEPARTED, RAINMAKER, GOOD SHEPHERD"
"✗ POLITICAL BELIEFSCONE, BLUE, GOOD SHEPHERD, LIBERALbest match: LEFT-LEANING, POLITICALLY (2/4)","NAMED ""APOLLO""GREEK/ROMAN GOD, SPACECRAFT, THEATER, FICTIONAL BOXER"


Contest,Solved,Group Acc,Partial-3,Partial-2
NYT Connections - 2025-11-25,✗,0%,0%,100%
NYT Connections - 2025-01-20,✗,0%,0%,100%
NYT Connections - 2025-07-18,✗,0%,25%,100%
NYT Connections - 2024-12-30,✗,0%,0%,50%
NYT Connections - 2023-10-13,✗,0%,75%,100%
NYT Connections - 2025-06-21,✗,0%,25%,100%
NYT Connections - 2025-09-04,✗,0%,25%,100%
NYT Connections - 2025-02-17,✗,0%,50%,100%
NYT Connections - 2024-12-07,✗,0%,25%,100%
NYT Connections - 2025-09-25,✗,0%,0%,75%


# RAG Evaluation

In [64]:
## Show Results for WordNet:
scores_with_RAG, summary_with_RAG = evaluate_run(
    raw_replies_with_RAG,
    all_actual_with_RAG
)

Predicted,Actual
"✗ DISNEY-RELATEDMICKEY MOUSE, BARBIE DREAMHOUSE, HAPPY MEAL, FLAMINGObest match: THINGS THAT ARE PINK (2/4)","SMALL-TIMEMICKEY MOUSE, RINKY-DINK, TRIVIAL, TWO-BIT"
"✗ CYCLES AND REPEATMERRY-GO-ROUND, RINKY-DINK, LOTTERY TICKET, YOUR HEADbest match: THINGS YOU CAN SCRATCH (2/4)","STARTING WITH OPTIMISTIC WORDSHAPPY MEAL, GLAD-HAND, MERRY-GO-ROUND, SUNNY-SIDE UP"
"✗ LIGHT AND EATINGSUNNY-SIDE UP, CALAMINE LOTION, TRIVIAL, TWO-BITbest match: SMALL-TIME (2/4)","THINGS THAT ARE PINKBARBIE DREAMHOUSE, CHERRY BLOSSOM, CALAMINE LOTION, FLAMINGO"
"✗ INSECT AND SKINBUG BITE, GLAD-HAND, CHERRY BLOSSOM, VINYL RECORDbest match: THINGS YOU CAN SCRATCH (2/4)","THINGS YOU CAN SCRATCHBUG BITE, LOTTERY TICKET, VINYL RECORD, YOUR HEAD"


Predicted,Actual
"✗ BODY PARTSSHOULDER, ELBOW, FOREARM, BENDbest match: CORNERS (2/4)","ASSOCIATED WITH POPEYESPINACH, ANCHOR, PIPE, FOREARM"
"✗ COOKING TERMSCOOKIE, SPINACH, CHEAT, HANDLEbest match: ___ SHEET (2/4)","CORNERSCROOK, ELBOW, BEND, ANGLE"
"✗ INVESTMENT TERMSBEAR, CHEAT, BALANCE, ASSUMEbest match: TAKE ON, AS A RESPONSIBILITY (2/4)","TAKE ON, AS A RESPONSIBILITYBEAR, SHOULDER, HANDLE, ASSUME"
"✗ SUPPORT AND CONNECTIONANCHOR, CROOK, PIPE, ANGLEbest match: ASSOCIATED WITH POPEYE (2/4)","___ SHEETCOOKIE, CHEAT, FITTED, BALANCE"


Predicted,Actual
"✓ VERBS OF PERCEPTIONSEE, SPOT, CATCH, NOTE","ELECTRIC ___SLIDE, GUITAR, BLANKET, EEL"
"✗ SLIDING MOVEMENTSSLIDE, RUN, STREAK, WHEELbest match: SEQUENCE (2/4)","PICK UP ONSEE, SPOT, CATCH, NOTE"
"✗ ANIMALSCOBRA, EEL, COW, STRINGbest match: YOGA BACKBENDS (2/4)","SEQUENCERUN, STRING, STREAK, SERIES"
"✗ MUSICAL TERMSGUITAR, STRING, BLANKET, SERIESbest match: ELECTRIC ___ (2/4)","YOGA BACKBENDSCOBRA, BRIDGE, COW, WHEEL"


Predicted,Actual
"✗ PINK-RELATED TERMSPINK, FILTER, MELT, COUGHbest match: LUNCH ORDERS (1/4)","LUNCH ORDERSCLUB, HERO, WRAP, MELT"
"✗ GAMING TERMSANTE, JEOPARDY, SETTLE, CLUBbest match: PAY, WITH “UP” (2/4)","NAMES FEATURING ""!""PINK, AIRPLANE, JEOPARDY, YAHOO"
"✗ WATER-RELATED TERMSWATER, BEANS, GRINDER, AIRPLANEbest match: USED TO MAKE COFFEE (3/4)","PAY, WITH “UP”PONY, ANTE, SETTLE, COUGH"
"✗ ANIMALSPONY, HERO, YAHOO, WRAPbest match: LUNCH ORDERS (2/4)","USED TO MAKE COFFEEGRINDER, WATER, BEANS, FILTER"


Predicted,Actual
"✗ CITIESLIMA, LAGOS, LUXOR, LINCOLNbest match: CITIES BEGINNING WITH “L” (3/4)","BEANSLIMA, PINTO, KIDNEY, FAVA"
"✗ POETIC FORMSLIMERICK, VERSE, RHYME, CREATORbest match: POETRY TERMS (2/4)","CITIES BEGINNING WITH “L”LIMERICK, LINCOLN, LAGOS, LUXOR"
"✗ HORSESSTALLION, PINTO, LINCOLN, DUDEbest match: “THE(E) ___” RAPPERS (2/4)","POETRY TERMSLINE, VERSE, RHYME, METER"
"✗ MEASUREMENTMETER, LINE, KIDNEY, FAVAbest match: BEANS (2/4)","“THE(E) ___” RAPPERSCREATOR, RAPPER, STALLION, DUDE"


...

Predicted,Actual
"✗ PAINTING AND DECORATINGAIRBRUSH, TOUCH UP, PHOTOSHOP, VINAIGRETTEbest match: CLEAN UP, AS A PHOTOGRAPH (3/4)","CLEAN UP, AS A PHOTOGRAPHAIRBRUSH, FIX, TOUCH UP, PHOTOSHOP"
"✗ FOOD ITEMSCANDY CANE, PAMPLEMOUSSE, CORNICHON, TINSELbest match: FRENCH FOOD WORDS (2/4)","FRENCH FOOD WORDSPAIN, PAMPLEMOUSSE, VINAIGRETTE, CORNICHON"
"✗ GADGETS AND TOOLSSOLAR PANEL, SPREADSHEET, FIX, HONEYCOMBbest match: THINGS WITH CELLS (3/4)","THINGS WITH CELLSHONEYCOMB, ORGANISM, SOLAR PANEL, SPREADSHEET"
"✗ EMOTIONS AND STATESPAIN, ANGEL, ORGANISM, STRING LIGHTSbest match: WHAT YOU MIGHT SEE ON A CHRISTMAS TREE (2/4)","WHAT YOU MIGHT SEE ON A CHRISTMAS TREECANDY CANE, ANGEL, TINSEL, STRING LIGHTS"


Predicted,Actual
"✗ TYPES OF VACATIONSVACATION, LEAVE, REST, SPAREbest match: TIME OFF (3/4)","BOWLING RESULTSDOUBLE, STRIKE, SPARE, TURKEY"
"✗ TYPES OF STRIKESBLOW, BREAK, STRIKE, DOUBLEbest match: BOWLING RESULTS (2/4)","FOLLOW A MEANDERING COURSESNAKE, WIND, WEAVE, CURVE"
"✓ WEAVING AND PATTERNSSNAKE, WIND, WEAVE, CURVE","TIME OFFBREAK, LEAVE, REST, VACATION"
"✗ TYPES OF TURKEYTURKEY, CAT, SWORD, GOLDbest match: ___FISH (3/4)","___FISHBLOW, CAT, SWORD, GOLD"


Predicted,Actual
"✗ VERBS RELATED TO MOVEMENTFLY, WINGS, STORM, BOWbest match: W.N.B.A. TEAMS (2/4)","LONG___HORN, LEGS, FELLOW, BOW"
"✗ ASSOCIATIONS OR CONNECTIONSASSOCIATE, AFFILIATE, FELLOW, RELATEbest match: THINK OF TOGETHER (3/4)","THINK OF TOGETHERASSOCIATE, RELATE, EQUATE, AFFILIATE"
"✗ SOURCES OF LIGHT OR ENERGYSUN, LIBERTY, HORN, CHUMbest match: W.N.B.A. TEAMS (2/4)","W.N.B.A. TEAMSSUN, WINGS, STORM, LIBERTY"
"✗ BAIT AND LURESBAIT, LURE, EQUATE, LEGSbest match: WAYS TO ATTRACT FISH (2/4)","WAYS TO ATTRACT FISHFLY, CHUM, BAIT, LURE"


Predicted,Actual
"✗ POLITICAL AFFILIATIONSLEFT, PROGRESSIVE, LIBERAL, GOOD SHEPHERDbest match: LEFT-LEANING, POLITICALLY (3/4)","ICE CREAM PARLOR ORDERSSPLIT, CUP, SHAKE, CONE"
"✗ SPACE RELATEDMARTIAN, SPACECRAFT, BLUE, SHAKEbest match: ICE CREAM PARLOR ORDERS (1/4)","LEFT-LEANING, POLITICALLYLEFT, PROGRESSIVE, BLUE, LIBERAL"
"✗ THEATRICAL TERMINOLOGYTHEATER, DEPARTED, SPLIT, RAINMAKERbest match: MATT DAMON MOVIES, WITH ""THE"" (2/4)","MATT DAMON MOVIES, WITH ""THE""MARTIAN, DEPARTED, RAINMAKER, GOOD SHEPHERD"
"✗ MYTHOLOGICAL REFERENCESGREEK/ROMAN GOD, FICTIONAL BOXER, CUP, CONEbest match: ICE CREAM PARLOR ORDERS (2/4)","NAMED ""APOLLO""GREEK/ROMAN GOD, SPACECRAFT, THEATER, FICTIONAL BOXER"


Contest,Solved,Group Acc,Partial-3,Partial-2
NYT Connections - 2025-11-25,✗,0%,0%,100%
NYT Connections - 2025-01-20,✗,0%,0%,100%
NYT Connections - 2025-07-18,✗,25%,25%,100%
NYT Connections - 2024-12-30,✗,0%,25%,75%
NYT Connections - 2023-10-13,✗,0%,25%,100%
NYT Connections - 2025-06-21,✗,0%,75%,100%
NYT Connections - 2025-09-04,✗,0%,50%,100%
NYT Connections - 2025-02-17,✗,25%,75%,100%
NYT Connections - 2024-12-07,✗,0%,25%,100%
NYT Connections - 2025-09-25,✗,0%,25%,75%


In [65]:
## Comparison: Baseline vs WordNet vs RAG
from IPython.display import display, HTML

metrics = ["gcr", "acc", "p3", "p2"]
labels  = ["Game Completion Rate", "Group Accuracy", "Partial-3 Rate", "Partial-2 Rate"]

def _delta_cell(a, b):
    diff = b - a
    color = "#66bb6a" if diff > 0 else ("#ef5350" if diff < 0 else "#aaa")
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "—")
    return (f'<td style="padding:6px 14px;text-align:center;color:{color};font-weight:bold">'
            f'{arrow} {abs(diff):.1%}</td>')

header = (
    '<tr style="color:#aaa;border-bottom:1px solid #555">'
    '<th style="padding:6px 14px;text-align:left">Metric</th>'
    '<th style="padding:6px 14px">Baseline (A)</th>'
    '<th style="padding:6px 14px">WordNet (B)</th>'
    '<th style="padding:6px 14px">RAG (C)</th>'
    '<th style="padding:6px 14px">Δ WordNet (B − A)</th>'
    '<th style="padding:6px 14px">Δ RAG (C − A)</th>'
    '</tr>'
)
rows = "".join(
    f'<tr>'
    f'<td style="padding:6px 14px">{lbl}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_no_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_with_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_with_RAG[m]:.1%}</td>'
    f'{_delta_cell(summary_no_hints[m], summary_with_hints[m])}'
    f'{_delta_cell(summary_no_hints[m], summary_with_RAG[m])}'
    f'</tr>'
    for m, lbl in zip(metrics, labels)
)

display(HTML(
    '<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    'Comparison: Baseline vs WordNet vs RAG</h2>'
    '<table style="border-collapse:collapse;font-family:monospace;font-size:0.95em">'
    f'{header}{rows}</table>'
    '<p style="color:#888;font-size:0.85em;font-family:monospace">'
    '▲ = Better than Baseline &nbsp;|&nbsp; ▼ = Worse than Baseline</p>'
))


Metric,Baseline (A),WordNet (B),RAG (C),Δ WordNet (B − A),Δ RAG (C − A)
Game Completion Rate,0.0%,0.0%,0.0%,— 0.0%,— 0.0%
Group Accuracy,7.5%,0.0%,5.0%,▼ 7.5%,▼ 2.5%
Partial-3 Rate,32.5%,22.5%,32.5%,▼ 10.0%,— 0.0%
Partial-2 Rate,92.5%,92.5%,95.0%,— 0.0%,▲ 2.5%
